# v27b — Stage B: white-box GCG with a joint loss (FRESH KERNEL, MPS)

**Before running:** restart the Mac, open only this notebook, swap < 2 GB. Qwen3-4B first; Llama-3.2-3B only if its smoke gate
projects ≤ 12 h. Resumable: one line per run in `results/v27/<MODEL>_B.jsonl`.

A 20-token adversarial suffix is appended to the seed injection. Loss = cross-entropy of the target string as the start of the reply
+ λ·softplus((s_d − τ_d)/σ_d) for the targeted frozen detector. GCG: top-64 tokens per position from the gradient, 32 candidates per
step, ≤ 150 steps. The fixed prefix is KV-cached. Every 10 steps the current suffix is **verified for real** through v26b's pipeline
(string → tokens → greedy generation → success rule → all detector scores); stop at the first verified win.

Runs: every instance gets C0 (λ 0), C-D3, C-D4, C-D6 (λ 1). The first 12 instances also get C-D3 and C-D4 at λ 0.3 and 3 (frontier).

Fidelity checks before any attack: **F1** cached forward == full forward; **F2** the differentiable detector scores == the frozen
numpy scores on the same prompt; **F3** the gradient is finite.

In [ ]:
MODEL = "qwen3-4b"        # <- "qwen3-4b" | "llama3.2-3b"
FORCE = False

import os, json, time, re, subprocess, hashlib, math, copy
import numpy as np, pandas as pd
import v27lib as L
MODEL_IDS = {"qwen3-4b": "Qwen/Qwen3-4B-Instruct-2507", "llama3.2-3b": "meta-llama/Llama-3.2-3B-Instruct"}
MODEL_PATH = os.environ.get("V27_MODEL_PATH") or MODEL_IDS[MODEL]
RES = "results/v27"; os.makedirs(RES, exist_ok=True)
REC, NFILE, VER, HB = f"{RES}/{MODEL}_B.jsonl", f"{RES}/{MODEL}_B_N.json", f"{RES}/{MODEL}_B_verify.json", f"{RES}/{MODEL}_B_heartbeat.txt"
MAX_LEN = int(os.environ.get("V26_DRY_MAXLEN", 1024))
DRY_N = int(os.environ.get("V27_DRY_N", 0)); DRY_STEPS = int(os.environ.get("V27_DRY_STEPS", 0))

def beat(s):
    with open(HB, "w") as fh: fh.write(f"{time.strftime('%Y-%m-%d %H:%M:%S')} | {MODEL} | {s}\n")
try:
    s = subprocess.run(["sysctl", "-n", "vm.swapusage"], capture_output=True, text=True, timeout=10).stdout
    m = re.search(r"used = ([\d.]+)([MG])", s); sw = float(m.group(1)) / (1024 if m.group(2) == "M" else 1) if m else None
    print("swap:", s.strip())
    if sw is not None and sw > 2.0 and not FORCE: raise RuntimeError(f"swap {sw:.1f} GB > 2 GB: restart the Mac first.")
except FileNotFoundError:
    pass

FZ = json.load(open("artifacts/v27_frozen.json"))
body = {k: v for k, v in FZ.items() if k != "sha256"}
assert hashlib.sha256(json.dumps(body, sort_keys=True).encode()).hexdigest() == FZ["sha256"], "I9: v27 frozen inputs changed"
assert L.sha_file("artifacts/v26_frozen.json") == FZ["v26_frozen_sha"], "I9: v26 detectors changed"
INST, TAU, CFG = FZ["instances"], FZ["tau"][MODEL], FZ["stageB"]
STEPS = DRY_STEPS or CFG["steps"]
beat("loaded")

In [ ]:
import torch, torch.nn.functional as Fn
from transformers import AutoTokenizer, AutoModelForCausalLM
tok = AutoTokenizer.from_pretrained(MODEL_PATH, token=os.environ.get("HF_TOKEN"))
model = AutoModelForCausalLM.from_pretrained(MODEL_PATH, dtype=torch.float16, attn_implementation="eager",
                                             token=os.environ.get("HF_TOKEN"))
DEV = "mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(DEV).eval()
for p in model.parameters(): p.requires_grad_(False)
RUN = L.Runner(model, tok, MODEL, max_len=MAX_LEN)        # verification path = v26b's hooks (inactive during GCG)
DET = L.Detectors("artifacts/v26_frozen.json", MODEL)
W = model.get_input_embeddings().weight                    # (V, H) fp16
V = W.shape[0]
NL = RUN.NL
# allowed suffix tokens: printable ASCII, not special, decodes to something
special = set(tok.all_special_ids)
ALLOWED = torch.zeros(V, dtype=torch.bool)
for t in range(min(V, len(tok))):
    if t in special: continue
    s_ = tok.decode([t])
    if s_ and s_.isascii() and s_.isprintable() and s_.strip() != "": ALLOWED[t] = True
ALLOWED = ALLOWED.to(DEV)
print(f"{MODEL_PATH} on {DEV} | vocab {V} | allowed suffix tokens {int(ALLOWED.sum())}")

## Problem setup per instance: prefix / suffix / tail / target, spans, KV cache

In [ ]:
def crop_to(cache, n):
    """Cut the KV cache back to n tokens. crop(-k) removes k tokens in transformers 4.x and 5.x alike."""
    k = cache.get_seq_length() - n
    if k > 0: cache.crop(-k)
    assert cache.get_seq_length() == n

class Problem:
    def __init__(self, inst, det, lam, b=8):
        self.inst, self.det, self.lam, self.b = inst, det, lam, b
        self.data0 = inst["twin"] + " " + inst["seed_piece"]
        MARK = "⁣"                                                         # invisible separator, used only to split strings
        f = RUN.fmt(inst["system"], RUN.user(inst["task"], self.data0 + MARK))
        k = f.index(MARK); self.pre_s, self.tail_s = f[:k], f[k + 1:]
        enc = tok(self.pre_s, add_special_tokens=False, return_offsets_mapping=True)
        self.pre = enc["input_ids"]; off = np.array(enc["offset_mapping"])
        self.tail = tok(self.tail_s, add_special_tokens=False)["input_ids"]
        self.target = tok(inst["target"], add_special_tokens=False)["input_ids"]
        self.P = len(self.pre)
        s0 = self.pre_s.index(inst["system"]); t0 = self.pre_s.index(inst["task"], s0 + len(inst["system"]))
        d0 = self.pre_s.index("<document>\n" + self.data0, t0) + len("<document>\n")
        tix = np.where((off[:, 0] < t0 + len(inst["task"])) & (off[:, 1] > t0))[0]
        dix = np.where(off[:, 1] > d0)[0]
        self.task_idx = torch.tensor(tix, device=DEV)
        self.data_start = int(dix.min())
        te = t0 + len(inst["task"]); self.anc = int(np.where(off[:, 0] < te)[0].max())
        ids = torch.tensor([self.pre], device=DEV)
        with torch.no_grad():
            o = model(input_ids=ids, use_cache=True, output_hidden_states=True)
        self.cache1_clean = o.past_key_values                                   # never used directly: copied per gradient step
        self.anchor = {li: o.hidden_states[li + 1][0, self.anc].detach() for li in range(NL)}
        with torch.no_grad():
            self.cacheb = model(input_ids=ids.repeat(b, 1), use_cache=True).past_key_values
        self.tail_t = torch.tensor(self.tail, device=DEV); self.tgt_t = torch.tensor(self.target, device=DEV)

    def text(self, suf):            # the data string the verifier sees
        return self.data0 + tok.decode(suf)

    def retok_ok(self, suf):
        full = tok(self.pre_s + tok.decode(suf) + self.tail_s, add_special_tokens=False)["input_ids"]
        return full == self.pre + list(suf) + self.tail

    def _forward(self, emb_suf, cache, need_attn):
        B = emb_suf.shape[0]; Ls = emb_suf.shape[1]
        rest = W[torch.cat([self.tail_t, self.tgt_t])].unsqueeze(0).expand(B, -1, -1)
        x = torch.cat([emb_suf, rest], 1)
        crop_to(cache, self.P)
        o = model(inputs_embeds=x, past_key_values=cache, use_cache=True, output_hidden_states=True,
                  output_attentions=need_attn)
        e = Ls + len(self.tail) - 1                                            # last prompt token (the 'end')
        logits = o.logits[:, e:e + len(self.target)].float()
        ce = Fn.cross_entropy(logits.reshape(-1, logits.shape[-1]), self.tgt_t.repeat(B), reduction="none").view(B, -1).mean(1)
        det_s = torch.zeros(B, device=DEV)
        if self.det is not None:
            data_idx = torch.cat([torch.arange(self.data_start, self.P, device=DEV), torch.arange(self.P, self.P + Ls, device=DEV)])
            vals = []
            for bi in range(B):
                hs = [h[bi, e] for h in o.hidden_states]
                rows = [a[bi, :, e, :] for a in o.attentions] if need_attn else None
                vals.append(DET.torch_score(self.det, hs, self.anchor, rows, (self.task_idx, data_idx)))
            det_s = torch.stack(vals)
        t, sg = TAU[self.det]["tau"] if self.det else 0.0, TAU[self.det]["sigma"] if self.det else 1.0
        loss = ce + self.lam * Fn.softplus((det_s - t) / max(sg, 1e-6))
        return loss, ce, det_s

    def grad(self, suf):
        x = W[torch.tensor(suf, device=DEV)].detach().float().requires_grad_(True)
        cache = copy.deepcopy(self.cache1_clean)                                # a backward pass leaves graph-linked slices behind
        loss, ce, s = self._forward(x.half().unsqueeze(0), cache, self.det == "D4")
        loss.sum().backward()
        del cache
        g = x.grad                                                              # (Ls, H) fp32
        G = torch.empty(len(suf), V, device=DEV)
        for a in range(0, V, 16384):                                            # gradient w.r.t. the one-hot, chunked
            G[:, a:a + 16384] = g @ W[a:a + 16384].float().T
        return G, float(loss.detach()), float(ce.detach()), float(s.detach()[0])

    def losses(self, cands):
        out = []
        for a in range(0, len(cands), self.b):
            chunk = cands[a:a + self.b]; pad = self.b - len(chunk)
            ids = torch.tensor(chunk + [chunk[-1]] * pad, device=DEV)
            with torch.no_grad():
                l, _, _ = self._forward(W[ids], self.cacheb, self.det == "D4")
            out += l[:len(chunk)].tolist()
        return out

def _init_suffix():
    """20 copies of one single-token filler that re-tokenises to itself inside the prompt (" !" first, as in GCG).
    The boundary context (seed piece ... suffix ... '</document>') is the same for every instance, so instance 0 decides it."""
    for cand in (" !", " x", " a", " the", " and", " of", "!", "x"):
        ids = tok(cand, add_special_tokens=False)["input_ids"]
        if len(ids) != 1 or not bool(ALLOWED[ids[0]]): continue
        init = ids * CFG["suffix_len"]
        if Problem(INST[0], None, 0.0, b=1).retok_ok(init): return init, cand
    raise RuntimeError("no single-token filler re-tokenises to itself; tell Claude")
INIT, FILLER = _init_suffix()
print(f"suffix: {len(INIT)} tokens of {FILLER!r}")

## F1–F3 fidelity (any failure stops the run)

In [ ]:
inst0 = INST[0]
for det in ("D3", "D4", "D6"):
    pb = Problem(inst0, det, 1.0)
    assert pb.retok_ok(INIT), "F1: the initial suffix does not re-tokenise to itself"
    # F1: cached forward == full forward (loss and detector score)
    with torch.no_grad():
        l_c, ce_c, s_c = [v.item() for v in pb._forward(W[torch.tensor([INIT], device=DEV)], copy.deepcopy(pb.cache1_clean), det == "D4")]
    full = torch.tensor([pb.pre + INIT + pb.tail + pb.target], device=DEV)
    with torch.no_grad():
        o = model(input_ids=full, output_hidden_states=True, output_attentions=(det == "D4"))
    e = pb.P + len(INIT) + len(pb.tail) - 1
    lg = o.logits[0, e:e + len(pb.target)].float()
    ce_f = Fn.cross_entropy(lg, pb.tgt_t).item()
    hs = [h[0, e] for h in o.hidden_states]; rows = [a[0, :, e, :] for a in o.attentions] if det == "D4" else None
    anc_f = {li: o.hidden_states[li + 1][0, pb.anc] for li in range(NL)}
    di = torch.arange(pb.data_start, pb.P + len(INIT), device=DEV)
    s_f = DET.torch_score(det, hs, anc_f, rows, (pb.task_idx, di)).item()
    assert abs(ce_c - ce_f) < 2e-2 * max(1, abs(ce_f)), f"F1 {det}: CE cached {ce_c:.4f} vs full {ce_f:.4f}"
    assert abs(s_c - s_f) < 2e-2 * max(TAU[det]["sigma"], 1e-3) + 1e-3, f"F1 {det}: score cached {s_c:.5f} vs full {s_f:.5f}"
    # F2: the differentiable score == the frozen numpy score from v26b's hook pipeline on the same prompt string
    v = RUN.evaluate(inst0["system"], inst0["task"], pb.text(INIT), "G1", inst0["target"], DET)
    assert abs(v["scores"][det] - s_f) < 2e-2 * max(TAU[det]["sigma"], 1e-3) + 1e-3, \
        f"F2 {det}: torch {s_f:.5f} vs frozen numpy {v['scores'][det]:.5f}"
    # F3: finite gradient
    G, *_ = pb.grad(INIT)
    assert torch.isfinite(G).all(), f"F3 {det}: non-finite gradient"
    print(f"  {det}: F1 CE {ce_c:.4f}=={ce_f:.4f}, score {s_c:+.4f}=={s_f:+.4f} | F2 numpy {v['scores'][det]:+.4f} | F3 ok")
    del pb, o
if DEV == "mps": torch.mps.empty_cache()
print("F1-F3 passed")
beat("fidelity passed")

## GCG

In [ ]:
def is_win(v, det): return bool(v["success"]) and (det is None or v["scores"][det] < TAU[det]["tau"])

def gcg(inst, det, lam, run_seed):
    t0 = time.perf_counter(); rng = np.random.default_rng(run_seed)
    pb = Problem(inst, det, lam); suf = list(INIT); trace, ver = [], []
    won = False
    for step in range(STEPS + 1):
        if step % CFG["verify_every"] == 0:
            v = RUN.evaluate(inst["system"], inst["task"], pb.text(suf), "G1", inst["target"], DET)
            ver.append(dict(step=step, success=v["success"], scores=v["scores"], response=v["response"][:300], suffix=tok.decode(suf)))
            if is_win(v, det): won = True; break
        if step == STEPS: break
        G, loss, ce, sd = pb.grad(suf)
        trace.append(dict(step=step, loss=loss, ce=ce, det=sd))
        G[:, ~ALLOWED] = float("inf")
        top = (-G).topk(CFG["topk"], dim=1).indices.cpu().numpy()
        cands = []
        for _ in range(4):                                     # up to 4 sampling rounds to fill the batch with valid candidates
            for _ in range(CFG["batch"]):
                c = list(suf); p = int(rng.integers(len(suf))); c[p] = int(top[p, int(rng.integers(CFG["topk"]))])
                if c != suf and pb.retok_ok(c): cands.append(c)
            if len(cands) >= CFG["batch"]: break
        cands = cands[:CFG["batch"]]
        if not cands: continue
        ls = pb.losses(cands); suf = cands[int(np.argmin(ls))]
    rec = dict(model=MODEL, idx=inst["idx"], target_detector=det, lam=lam, won=won, steps=step,
               any_success=any(x["success"] for x in ver), seconds=time.perf_counter() - t0, verifications=ver, trace=trace[::5])
    del pb
    if DEV == "mps": torch.mps.empty_cache()
    return rec

done = set()
if os.path.exists(REC):
    for line in open(REC): r = json.loads(line); done.add((r["idx"], r["target_detector"], r["lam"]))
PRIMARY = [(None, 0.0), ("D3", CFG["lam_primary"]), ("D4", CFG["lam_primary"]), ("D6", CFG["lam_primary"])]
FRONT = [(d, l) for d in ("D3", "D4") for l in CFG["frontier_lams"]]
def runs_for(i): return PRIMARY + (FRONT if i < CFG["frontier_n"] else [])
def run_one(i, det, lam):
    rec = gcg(INST[i], det, lam, 27_500_000 + i * 100 + (["None", "D3", "D4", "D6"].index(str(det))) * 10 + int(lam * 3))
    with open(REC, "a") as fh: fh.write(json.dumps(rec) + "\n")
    done.add((i, det, lam)); return rec

## Smoke gate → N (pre-registered: worst-case cost of 4 primary runs per instance + the 12-instance frontier must fit 12 h; N ≥ 24)

In [ ]:
if os.path.exists(NFILE):
    N = json.load(open(NFILE))["N"]; print(f"N already fixed at {N}")
else:
    pb = Problem(INST[0], "D4", 1.0)
    G, *_ = pb.grad(INIT); _ = pb.losses([INIT] * CFG["batch"])            # warm-up
    if DEV == "mps": torch.mps.synchronize()
    t = time.perf_counter()
    for _ in range(3):
        G, *_ = pb.grad(INIT); _ = pb.losses([INIT] * CFG["batch"])
    if DEV == "mps": torch.mps.synchronize()
    t_step = (time.perf_counter() - t) / 3
    t0 = time.perf_counter(); RUN.evaluate(INST[0]["system"], INST[0]["task"], pb.text(INIT), "G1", INST[0]["target"], DET); t_ver = time.perf_counter() - t0
    del pb
    t_run = t_step * CFG["steps"] + (CFG["steps"] // CFG["verify_every"] + 1) * t_ver
    budget = FZ["budget"]["stageB_hours"] * 3600
    N = int(min(FZ["budget"]["n_max"], math.floor((budget / t_run - len(FRONT) * CFG["frontier_n"]) / len(PRIMARY))))
    if DRY_N: N = DRY_N
    print(f"step {t_step:.2f} s, verify {t_ver:.1f} s -> worst case {t_run/60:.1f} min per run -> N = {N}")
    if N < FZ["budget"]["n_min"] and not DRY_N:
        raise RuntimeError(f"N = {N} < {FZ['budget']['n_min']}: stage B does not fit 12 h on {MODEL}. "
                           "Pre-registered: on Llama this means stage B is reported black-box only; on Qwen, stop and tell Claude.")
    json.dump(dict(N=N, seconds_per_step=t_step, seconds_per_verify=t_ver, worst_minutes_per_run=t_run / 60,
                   fixed=time.strftime("%Y-%m-%d %H:%M:%S")), open(NFILE, "w"), indent=1)
beat(f"N = {N}")

## Full run — resumable

In [ ]:
todo = [(i, d, l) for i in range(N) for d, l in runs_for(i) if (i, d, l) not in done]
print(f"{len(todo)} runs to do")
t_all, k = time.perf_counter(), 0
for i, d, l in todo:
    rec = run_one(i, d, l); k += 1
    el = time.perf_counter() - t_all
    print(f"  [{i:02d} {str(d):4s} λ={l:<3}] {'WIN ' if rec['won'] else 'lose'} at step {rec['steps']:3d} in {rec['seconds']:5.0f} s | "
          f"{k}/{len(todo)} | left ~{el / k * (len(todo) - k) / 3600:.1f} h", flush=True)
    beat(f"run {k}/{len(todo)}: instance {i} {d} {l}")
print("stage B complete")

## I7 — re-verify every win from scratch

In [ ]:
recs = [json.loads(l) for l in open(REC)]
out, bad = [], 0
for r in recs:
    if not r["won"]: continue
    inst = INST[r["idx"]]; w = r["verifications"][-1]
    data = inst["twin"] + " " + inst["seed_piece"] + w["suffix"]
    v = RUN.evaluate(inst["system"], inst["task"], data, "G1", inst["target"], DET)
    ok = is_win(v, r["target_detector"]); bad += (not ok)
    out.append(dict(idx=r["idx"], target_detector=r["target_detector"], lam=r["lam"], verified=bool(ok)))
json.dump(out, open(VER, "w"), indent=1)
print(f"I7: {len(out)} wins re-verified, {bad} did not reproduce (counted as losses in v27c)")
RUN.remove(); del model; import gc; gc.collect()
if DEV == "mps": torch.mps.empty_cache()
beat("done; model released")